<a href="https://colab.research.google.com/github/leticialindona/prova_pestana/blob/main/Guia_rapido_consulta_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ETAPA 1

# Notebook de Consulta Rápida: Extração e Análise de Dados

Este é o guia de bolso para a prova prática, desenhado para buscas rápidas (Ctrl+F).
Siga o mapa abaixo para pular direto para a seção necessária.

| Se a questão pede | Vá para |
| --- | --- |
| Bibliotecas e pacotes iniciais | §00 |
| Ler arquivo de dados bruto ou limpo | §01 |
| Dimensões, primeiras linhas e tipos | §02 |
| Quantos valores nulos existem | §03 |
| Remover registros repetidos | §04 |
| Arrumar grafias diferentes de categoria | §05 |
| Converter texto em data/hora | §06 |
| Converter texto em número | §07 |
| Tratar vazios e descartar linhas | §08 |
| Calcular percentual ou taxa | §09 |
| Agrupar e resumir por uma categoria | §10 |
| Agrupar por combinação de categorias | §11 |
| Ranking, top 10, maiores valores | §12 |
| Filtrar pedaço específico da base | §13 |
| Extrair dia, hora ou dia da semana | §14 |
| Comparar categorias (barras em pé) | §15 |
| Comparar categorias longas (barras deitadas) | §16 |
| Evolução no tempo (linhas) | §17 |
| Correlação e tendência (pontos) | §18 |
| Treino e teste (separar base) | §19 |
| Modelos de Classificação (qual usar) | §20 |
| Modelos de Regressão (qual usar) | §21 |
| Modelos de Clusterização | §22 |
| Matriz de confusão e métricas | §23 |
| MAE e R² (avaliar regressão) | §24 |
| Importância de variáveis (árvore) | §25 |
| Mudar ponto de corte (threshold) | §26 |
| Erro: não encontrou arquivo | §27 |
| Erro: coluna não existe | §28 |
| Erro: não converte tipo | §29 |
| Frases prontas para justificar escolhas | §30 |

# §00 · Imports

`Ctrl+F:` importar, dependências, bibliotecas, plt, pd

```python
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np # (não caiu nas aulas)

```

# §01 · Carregar o CSV

`Ctrl+F:` ler arquivo, importar csv, carregar base, read_csv

```python
# TROQUE: o nome do arquivo
bruto = pd.read_csv("dados/publicacoes_brutas.csv")
# se der erro de separador: pd.read_csv("...", sep=";")

```

# §02 · Olhar a base antes de mexer (head, shape, dtypes)

`Ctrl+F:` inspecionar, tamanho, quantas linhas, tipos de dados, head, shape, dtypes

```python
print("Tamanho (linhas, colunas):", bruto.shape)
print("\nTipos das colunas:\n", bruto.dtypes)
bruto.head()

```

# §03 · Valores ausentes, mostrando só as colunas que têm ausência

`Ctrl+F:` nulos, vazios, ausentes, faltantes, isna, isna().sum()

O `.isna().sum()` puro lista todas as colunas; guarde em uma variável e filtre apenas os maiores que zero.

```python
ausentes = bruto.isna().sum()
# TROQUE: a variável da tabela, se não for bruto
ausentes_com_problema = ausentes[ausentes > 0]
print(ausentes_com_problema)

```

# §04 · Duplicatas

`Ctrl+F:` duplicidade, duplicata, registro repetido, drop_duplicates, por id

`drop_duplicates()` sem argumento só apaga linhas idênticas em todas as colunas; se pedir por identificador, use `subset`.

```python
# TROQUE: a coluna de identificador
print("duplicadas:", bruto.duplicated(subset="id_publicacao").sum())
limpo = bruto.drop_duplicates(subset="id_publicacao", keep="first").copy()
# se pedir linha inteira repetida: bruto.drop_duplicates()

```

# §05 · Padronizar texto e categoria (grafias diferentes da mesma coisa)

`Ctrl+F:` padronizar, limpar texto, lower, strip, replace, grafias diferentes

Olhe o `value_counts()` para descobrir os erros de digitação antes de montar o dicionário de equivalências.

```python
# TROQUE: a coluna a ser padronizada
limpo["tema"] = limpo["tema"].str.strip().str.lower()
print("Antes:", limpo["tema"].value_counts(dropna=False))

# TROQUE: o dicionário com os erros encontrados no value_counts
equivalencias = {"saúde": "saude"}
limpo["tema"] = limpo["tema"].replace(equivalencias)
print("Depois:", limpo["tema"].unique())

```

# §06 · Converter data

`Ctrl+F:` converter data, transformar em data, to_datetime, format, dayfirst, NaT

Sem `format` o pandas adivinha pela primeira linha e mata como `NaT` o que não encaixa, e `dayfirst=True` na coluna toda troca dia e mês das datas ISO. Converta com `format="mixed", dayfirst=True` e sempre confira o `.min()` e `.max()` para garantir que o período faz sentido.

```python
# TROQUE: o nome da coluna de data
limpo["data_publicacao"] = pd.to_datetime(
  limpo["data_publicacao"], format="mixed", dayfirst=True, errors="coerce"
)

print("Ausentes gerados:", limpo["data_publicacao"].isna().sum())
print("Período: de", limpo["data_publicacao"].min(), "até", limpo["data_publicacao"].max())

```

# §07 · Coluna numérica que veio como texto

`Ctrl+F:` veio como texto, to_numeric, converter número, ValueError

```python
# TROQUE: o nome da coluna
limpo["alcance"] = pd.to_numeric(limpo["alcance"], errors="coerce")
limpo.dtypes

```

# §08 · Ausências e valores inválidos

`Ctrl+F:` remover linhas vazias, descartar, tratar nulos, dropna

```python
# TROQUE: as colunas onde a ausência não é tolerada
limpo = limpo.dropna(subset=["tema"]).copy()

ou

print(limpo[limpo["alcance"].isna()])
limpo = limpo.dropna(subset=["alcance", "salvamentos"])

```

# §09 · Criar coluna calculada (taxa, percentual)

`Ctrl+F:` calcular taxa, percentual, criar coluna, proporção

```python
# TROQUE: o nome da nova coluna e a fórmula matemática
limpo["taxa_utilidade_pct"] = (
    (limpo["compartilhamentos"] + limpo["salvamentos"]) / limpo["alcance"] * 100
)
limpo.head()
```

# §10 · Tabela resumo por UMA coluna (groupby, agg, reset_index, sort_values)

`Ctrl+F:` tabela resumo, agrupar por, média por, mediana por, groupby, agg

```python
# TROQUE: a coluna do agrupamento, o nome da nova coluna e a operação ("mean", "median", "sum", "count")
resumo = limpo.groupby("tema").agg(
    qtd_publicacoes=("id_publicacao", "count"),
    mediana_utilidade=("taxa_utilidade_pct", "median")
).reset_index().sort_values("mediana_utilidade", ascending=False)
print(resumo.to_string(index=False)) # (não caiu nas aulas)

```

# §11 · Tabela resumo por DUAS colunas

`Ctrl+F:` combinar categorias, agrupar por duas, cruzamento

```python
# TROQUE: as colunas da lista do groupby e os nomes/operações do agg
resumo_duplo = limpo.groupby(["tema", "formato"]).agg(
    publicacoes=("id_publicacao", "count"),
    mediana_engajamento=("taxa_engajamento_pct", "median") # usa a mesma logica para engajamento se precisar
).reset_index().sort_values("mediana_engajamento", ascending=False)
print(resumo_duplo.to_string(index=False)) # (não caiu nas aulas)

```

# §12 · Os N maiores, ordenar, ranking

`Ctrl+F:` maiores, top 10, ranking, ordenar, sort_values

```python
# TROQUE: a coluna que define o ranking e quantas linhas exibir no head
top5 = limpo.sort_values("taxa_engajamento_pct", ascending=False).head(5).copy()
# TROQUE: as colunas que importam na exibição
print(top5[["id_publicacao", "tema", "taxa_engajamento_pct"]])

```

# §13 · Filtrar linhas

`Ctrl+F:` apenas, somente, filtrar, recorte, condição

```python
# TROQUE: a condição do filtro
recorte_reels = limpo[limpo["formato"] == "reel"].copy()
# se for filtro composto: limpo[(limpo["formato"] == "reel") & (limpo["alcance"] > 1000)].copy() (não caiu nas aulas)

```

# §14 · Data: dia, hora, dia da semana

`Ctrl+F:` extrair dia, só a data, hora, dt.date

```python
# TROQUE: as colunas de origem e destino
limpo["dia_publicacao"] = limpo["data_publicacao"].dt.date
limpo["hora"] = limpo["data_publicacao"].dt.hour # (não caiu nas aulas)
limpo["dia_semana"] = limpo["data_publicacao"].dt.dayofweek # (não caiu nas aulas)

```

# §15 · Gráfico de barras vertical, com título, nome de eixo e fonte

`Ctrl+F:` gráfico de barras, barras em pé, plotar barras, ax.bar

```python
fig, ax = plt.subplots(figsize=(8, 5))
# TROQUE: x e y (nomes das colunas da tabela que vai ser plotada)
ax.bar(resumo["tema"], resumo["mediana_utilidade"], color="#3b6ea5")

ax.set_title("Mediana da Taxa de Utilidade por Tema")
ax.set_xlabel("Tema da publicação")
ax.set_ylabel("Taxa de Utilidade (%)")
ax.tick_params(axis="x", rotation=45)
# TROQUE: o texto da fonte
fig.text(0.01, -0.05, "Fonte: dados sintéticos do Festival ViraBairro (2026)", fontsize=8, color="gray")

fig.tight_layout()
plt.show()

```

# §16 · Gráfico de barras horizontal

`Ctrl+F:` barras deitadas, gráfico horizontal, barh, variáveis mais importantes

```python
fig, ax = plt.subplots(figsize=(8, 5))
# TROQUE: y e largura das barras (geralmente usado para importâncias de árvore)
ax.barh(resumo["tema"], resumo["mediana_utilidade"], color="#3b6ea5") # (não caiu nas aulas)

ax.set_title("Mediana da Taxa de Utilidade por Tema")
ax.set_xlabel("Taxa de Utilidade (%)")
ax.set_ylabel("Tema da publicação")
# ax.invert_yaxis() # útil para deixar o maior no topo (não caiu nas aulas)

fig.tight_layout()
plt.show()

ou

resumo2["combinacao"] = resumo2["tema"] + " / " + resumo2["formato"]
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(resumo2["combinacao"][::-1], resumo2["mediana_pct"][::-1], color="#3b6ea5")
ax.set_title("Quais combinações de tema e formato engajam mais?")
ax.set_xlabel("Mediana da taxa de engajamento (%)")
ax.set_ylabel("Tema / formato")
fig.text(0.01, -0.02, "Fonte: dados sintéticos do Festival ViraBairro (2026)", fontsize=8)
fig.tight_layout()
plt.show()

```

# §17 · Gráfico de linhas ao longo do tempo

`Ctrl+F:` evolução, ao longo do tempo, tendência diária, gráfico de linha, plot

```python
fig, ax = plt.subplots(figsize=(9, 5))
# TROQUE: a base temporária e os eixos x/y
ax.plot(diario["dia_publicacao"], diario["media_engajamento"], marker="o", color="#e85d04")

ax.set_title("Evolução da Taxa Média de Engajamento por Dia")
ax.set_xlabel("Data de publicação")
ax.set_ylabel("Engajamento médio (%)")
ax.tick_params(axis="x", rotation=45)
fig.text(0.01, -0.05, "Fonte: dados sintéticos do Festival ViraBairro (2026)", fontsize=8, color="gray")

fig.tight_layout()
plt.show()

ou

tabela_diaria = (
    limpo.groupby("dia_publicacao")
    .agg(publicacoes=("id_publicacao", "count"),
         media_pct=("taxa_engajamento_pct", "mean"),
         alcance_total=("alcance", "sum"))
    .reset_index()
    .sort_values("dia_publicacao")
)
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(tabela_diaria["dia_publicacao"], tabela_diaria["media_pct"], marker="o", markersize=3)
ax.set_title("Como a taxa média de engajamento variou dia a dia")
ax.set_xlabel("Dia da publicação")
ax.set_ylabel("Taxa média de engajamento (%)")
ax.tick_params(axis="x", rotation=45)
fig.text(0.01, -0.02, "Fonte: dados sintéticos do Festival ViraBairro (2026)", fontsize=8)
fig.tight_layout()
plt.show()

```

# §18 · Dispersão com linha de referência

`Ctrl+F:` real versus previsto, correlação, dispersão, pontos, scatter

```python
fig, ax = plt.subplots(figsize=(6, 6))
# TROQUE: x (valores reais) e y (previsões do modelo)
ax.scatter(y_teste, previsoes, alpha=0.6, color="#2a9d8f")

# Linha de referência
limites = [min(y_teste.min(), previsoes.min()), max(y_teste.max(), previsoes.max())] # (não caiu nas aulas)
ax.plot(limites, limites, color="red", linestyle="--") # (não caiu nas aulas)

ax.set_title("Engajamento: Valores Reais vs. Previstos")
ax.set_xlabel("Taxa Real (%)")
ax.set_ylabel("Previsão do Modelo (%)")
fig.tight_layout()
plt.show()

```

ETAPA 2

# §19 · Regressão, classificação ou clusterização?

`Ctrl+F:` qual modelo usar, tipo de aprendizado, decidir modelo, prever, agrupar

| O que você quer fazer? | Tipo de Aprendizado | Exemplo prático |
| --- | --- | --- |
| Prever um número (ex: estimar a taxa, curtidas) | Regressão | Estimar a `taxa_engajamento_pct` |
| Prever uma categoria ou decisão (ex: sim/não, viralizou) | Classificação | Decidir se a peça `mereceu_divulgacao_adicional` |
| Encontrar perfis parecidos na base sem rótulo prévio | Clusterização | Separar clientes/publicações em grupos de perfil |

# §20 · Montar X e y, get_dummies e vazamento

`Ctrl+F:` criar X e y, preparar dados, texto para número, vazamento, get_dummies

O alvo (y) e qualquer coluna calculada a partir dele não podem entrar no X, senão o modelo decora o resultado (vazamento).

```python
# TROQUE: a coluna alvo (y) e as colunas disponíveis antes da publicação (X)
y = limpo["taxa_engajamento_pct"]
X_base = limpo[["tema", "formato", "tamanho_legenda", "hora", "dia_semana"]]
X = pd.get_dummies(X_base)

ou

# TROQUE: coluna alvo (y) e colunas de feature em X
y = limpo["taxa_engajamento_pct"]
X = limpo[["hora", "dia_semana", "alcance"]].copy()
X = pd.get_dummies(X.join(limpo[["tema", "formato"]]), columns=["tema", "formato"])
X.nunique()
```

# §21 · Treino e teste

`Ctrl+F:` separar treino, teste, train_test_split, split, stratify

O parâmetro `stratify=y` preserva a proporção da classe rara, mas deve ser usado apenas em classificação, nunca em regressão.

```python
from sklearn.model_selection import train_test_split
# TROQUE: test_size e remova "stratify=y" se o modelo for de regressão
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

```

# §22 · Regressão: estimar um número (MAE, R², modelo bobo)

`Ctrl+F:` regressão linear, árvore de regressão, avaliar erro, mae, r2, modelo bobo

```python
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
# TROQUE: o modelo (ou use DecisionTreeRegressor(max_depth=4, random_state=42))
modelo = LinearRegression()
modelo.fit(X_treino, y_treino)
previsoes = modelo.predict(X_teste)
previsoes_bobas = np.full(len(y_teste), y_treino.mean())

ou

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

# TROQUE: modelo; alternativa: DecisionTreeRegressor(max_depth=5, random_state=42)
modelo = LinearRegression()
modelo.fit(X_treino, y_treino)
previsoes = modelo.predict(X_teste)
print("MAE:", mean_absolute_error(y_teste, previsoes))
print("R2:", r2_score(y_teste, previsoes))
bobo = np.full(len(y_teste), y_treino.mean())
print("MAE bobo:", mean_absolute_error(y_teste, bobo))

```

# §23 · Criar o rótulo por percentil

`Ctrl+F:` criar alvo, percentil, corte, quantile, astype(int)

```python
# TROQUE: o limite (0.75 para percentil 75) e o nome da coluna alvo
corte = limpo["taxa_engajamento_pct"].quantile(0.75)
limpo["mereceu_divulgacao_adicional"] = (limpo["taxa_engajamento_pct"] > corte).astype(int)
# y = limpo["mereceu_divulgacao_adicional"]

```

# §24 · Classificação: treinar e comparar modelos

`Ctrl+F:` treinar classificador, regressão logística, árvore de decisão, prever classe

```python
from sklearn.linear_model import LogisticRegression
# TROQUE: o classificador. Alternativa: DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced")
modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_treino, y_treino)
decisao = modelo.predict(X_teste)
probabilidade = modelo.predict_proba(X_teste)[:, 1]

```

# §25 · Métricas e matriz de confusão

`Ctrl+F:` avaliar classificação, matriz de confusão, precisão, recall, f1, accuracy

```python
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
# TROQUE: os nomes das variáveis do teste e predição, se diferentes
print("Matriz de confusão:\n", confusion_matrix(y_teste, decisao))
print("acuracia:", accuracy_score(y_teste, decisao))
print("Precisão:", precision_score(y_teste, decisao, zero_division=0))
print("Recall:", recall_score(y_teste, decisao, zero_division=0))
print("F1:", f1_score(y_teste, decisao, zero_division=0))

```

# §26 · Ajustar o corte (threshold)

`Ctrl+F:` mudar corte, threshold, predict_proba, ponto de corte, ajustar probabilidade

O modelo decide no 0,5 por padrão; alterar o corte muda o trade-off entre falso positivo e falso negativo.

```python
# TROQUE: o valor do corte desejado (ex: 0.30)
probabilidade = modelo.predict_proba(X_teste)[:, 1]
nova_decisao = (probabilidade >= 0.30).astype(int)
print("Novo Recall:", recall_score(y_teste, nova_decisao, zero_division=0))

```

# §27 · Importâncias e coeficientes

`Ctrl+F:` importância, variables, coeficientes, feature_importances, quem impactou mais

```python
importancias = pd.DataFrame({
    "feature": X_treino.columns,
    # TROQUE: use modelo.feature_importances_ (árvore) ou modelo.coef_[0] (linear/logística)
    "peso": modelo.feature_importances_
}).sort_values("peso", ascending=False)
print(importancias.head(5))

ou

# TROQUE: modelo treinado (árvore) e nomes das colunas de X
importancias = pd.Series(modelo.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(importancias.index, importancias.values, color="#3b6ea5")
ax.set_title("Quais variáveis mais pesaram na decisão da árvore")
fig.tight_layout()
plt.show()
# regressão linear usa modelo.coef_ em vez de feature_importances_

```

# §28 · Clusterização com KMeans

`Ctrl+F:` cluster, kmeans, grupos, padronizar, separar grupos

Esquecer de padronizar fará com que a coluna de maior grandeza domine os clusters sozinha.

```python
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
# TROQUE: as colunas escolhidas e o número de clusters (k)
X_padronizado = StandardScaler().fit_transform(X)
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
grupos = kmeans.fit_predict(X_padronizado)
# print("Silhueta:", silhouette_score(X_padronizado, grupos)) # (não caiu nas aulas) -> ops, confere Aula 13

ou

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# TROQUE: colunas numéricas e número de clusters
X_num = limpo.select_dtypes("number")
X_padronizado = StandardScaler().fit_transform(X_num)
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
limpo["cluster"] = kmeans.fit_predict(X_padronizado)
limpo.groupby("cluster")[X_num.columns].mean()
```

# §29 · Deu erro e Troubleshooting

`Ctrl+F:` erro, consertar, troubleshoot, problema silencioso, bug

| Mensagem / O que aconteceu | O que é | Vá para § |
| --- | --- | --- |
| `FileNotFoundError` | O caminho do arquivo está errado ou a pasta `dados/` faltou. | §01 |
| `KeyError: 'nome'` | Você digitou uma coluna que não existe ou tem espaço oculto. | §02 |
| `ValueError: could not convert string to float` | Faltou converter texto para número (get_dummies) no `X`. | §20 |
| `ValueError: Input contains NaN` | O modelo não aceita vazios; faltou tratar nulos antes do `fit`. | §08 |
| Acurácia de 95%, mas Recall de 0% | A classe rara dominou; erro silencioso. Faltou balancear peso. | §24 |
| Erro R² negativo ou 1.0 (perfeito demais) | Vazamento de dados! O alvo (y) acabou ficando dentro do X. | §20 |
| MAE no treino pequeno, MAE no teste enorme | O modelo decorou (overfitting). Diminua o `max_depth`. | §22 |
| Matriz de confusão não soma o total real | Algum `dropna()` mudou o tamanho do `X` após separar o `y`. | §21 |
| Clusters não fazem sentido de negócio | Falta de padronização (StandardScaler) antes de rodar o K-Means. | §28 |
| `NameError: pd is not defined` | Você pulou a primeira célula de importar pacotes. | §00 |

# §30 · Frases prontas (Respostas Markdown)

`Ctrl+F:` justificar, texto pronto, preencher, responder prova, interpretação

1. A mediana observada foi de ____, o que significa que metade das publicações teve esse valor ou um valor menor.
2. O modelo erra, em média, ____ pontos na taxa ao realizar uma previsão em dados não vistos (MAE).
3. O modelo obteve um R² de ____, indicando a proporção da variação dos dados que é explicada por ele.
4. É importante destacar que as variáveis mais importantes revelam associação nos dados, mas não provam causalidade mecânica sobre o resultado.
5. O tamanho da amostra (____ registros) é uma limitação, restringindo conclusões a nichos muito específicos.
6. Devido ao empate técnico entre as categorias ____ e ____, ambas deveriam ser priorizadas ou testadas no longo prazo.
7. Decidi remover os registros sem informação porque preenchê-los artificialmente enviesaria a decisão de comunicação final.
8. Justifico a mudança no corte para ____ porque, para a campanha, deixar passar um viral (falso negativo) é pior do que alertar peças medianas (falso positivo).
9. Removi a variável ____ do treino porque ela sofre alteração após a publicação, o que causaria vazamento de dados no cenário preditivo.
10. O modelo foi treinado com o escopo atual, limitando sua generalização caso o comportamento do público mude drasticamente.